# 09 · Assemble a TS from a core spec and fresh fragments

Notebook 05 transferred a geometry you already had. Here there is none: state the reacting core as
distances/angles or coordinates, and hang real fragments around it.

- `rx.match` resolves a SMARTS to atom indices
- `fix={(i,j): d, (i,j,k): θ}` states the core as numbers, `fix={i: (x,y,z)}` as coordinates
- `.`-disconnected fragments are tethered at van der Waals contact, so nothing drifts off

In [ ]:
import rxembed as rx
import xyzrender
from rdkit import Chem, RDLogger

RDLogger.DisableLog("rdApp.*")
rx.set_verbose("INFO")

## 1 · The core as numbers

An SN2 core from chemical intuition: F⁻ at 2.0 Å from the carbon, C-Cl half-broken at 2.3 Å, the three
collinear. Two `.`-disconnected SMILES. Every returned conformer must satisfy the numeric `fix`; `.measure()`
reads it back.


In [ ]:
mol = Chem.AddHs(Chem.MolFromSmiles("[F-].CCCCl"))  # fluoride + 1-chloropropane, two fragments
(f,), (c, cl) = rx.match(mol, "[F-]"), rx.match(mol, "[CH2]Cl")

ens = rx.embed(mol, fix={(f, c): 2.0, (c, cl): 2.3, (f, c, cl): 180.0}).mc().minimize().prune()
print(f"{len(ens)} conformers")
for label, atoms in (("F...C", (f, c)), ("C...Cl", (c, cl)), ("F-C-Cl", (f, c, cl))):
    print(f"  {label:7s} {ens.measure(atoms)['mean']:.2f}")

for report in ens.check(frozen=[f, c, cl]).values():
    print(report.summary())
xyzrender.render(
    ens.mol,
    no_hy=True,
    ts_bonds=[(f + 1, c + 1), (c + 1, cl + 1)],
)

## 2 · The core as coordinates

`fix={i: (x, y, z)}` places those atoms exactly and searches everything else. The core is Kabsch-grafted
back, so the measured distances match the input to the milli-ångström.


In [ ]:
mol = Chem.AddHs(Chem.MolFromSmiles("[F-].c1ccccc1CCl"))  # fluoride + benzyl chloride
(f,), (c, cl) = rx.match(mol, "[F-]"), rx.match(mol, "[CH2]Cl")
core = {f: (0.0, 0.0, 0.0), c: (2.0, 0.0, 0.0), cl: (4.3, 0.0, 0.0)}  # a linear core, explicit xyz

ens = rx.embed(mol, fix=core).mc()
print(f"{len(ens)} conformers | grafted F...C {ens.measure((f, c))['mean']:.3f} A")

for report in ens.check(frozen=[f, c, cl]).values():
    print(report.summary())
xyzrender.render(ens.align().mol, no_hy=True)

## 3 · A core across three fragments

A core is a set of indices, so it can pin an atom in each of several `.`-joined fragments at once: a base,
the substrate, and a water microsolvating the leaving chloride. What the core does not pin is tethered.


In [ ]:
mol = Chem.AddHs(Chem.MolFromSmiles("[F-].CCCCCl.O"))  # base . substrate . water
(f,), (c, cl) = rx.match(mol, "[F-]"), rx.match(mol, "[CH2]Cl")
(o,) = rx.match(mol, "[OX2H2]")
core = {f: (0.0, 0.0, 0.0), c: (2.0, 0.0, 0.0), cl: (4.3, 0.0, 0.0), o: (5.6, 1.8, 0.0)}

ens = rx.embed(mol, fix=core).mc().minimize().prune()
print(f"{len(ens)} conformers over {len(Chem.GetMolFrags(mol))} fragments")
print(f"  Cl...O {ens.measure((cl, o))['mean']:.2f} A")

for report in ens.check(frozen=[f, c, cl, o]).values():
    print(report.summary())
xyzrender.render(
    ens.mol,
    no_hy=True,
    ts_bonds=[(f + 1, c + 1), (c + 1, cl + 1)],
)

## 4 · One core spec, several substrates

The same SN2 core across three substrates. Each returns a conformer-searched, GFN-FF-ranked ensemble of the
assembled complex with the forming and breaking distances held. GFN-FF requires `xtb` on `$XTB_EXE`.


In [ ]:
CORE = {(0, 1): 2.0, (1, 2): 2.3, (0, 1, 2): 180.0}  # keyed by position in (f, c, cl)

for smi in ["[F-].CCCCCl", "[F-].CC(C)CCl", "[F-].c1ccccc1CCCl"]:
    mol = Chem.AddHs(Chem.MolFromSmiles(smi))
    (f,), (c, cl) = rx.match(mol, "[F-]"), rx.match(mol, "[CH2]Cl")
    at = (f, c, cl)
    fix = {tuple(at[k] for k in key): v for key, v in CORE.items()}
    ens = rx.embed(mol, fix=fix).mc(preset="rapid").minimize().prune()
    ens = ens.filter("geometry", frozen=at).score("gfnff")
    dE = sorted(ens.relative().values())
    print(f"  {smi:22s} {ens.n:3d} conformers | GFN-FF spread {dE[-1] - dE[0]:4.1f} kcal/mol")

## The three spellings

A core is a set of indices plus one of:

- `fix={(i,j): d, (i,j,k): θ}` numbers (§1)
- `fix={i: (x,y,z)}` coordinates (§2, §3)
- `template=(reference, {i: ref_i})` coordinates pulled from a geometry you have (notebook 05)

`constrain={...}` is the same again but soft, a bias a real energy may overrule. `rx.minimize(source,
fix=...)` relaxes a geometry you already hold toward the core instead of conformer-searching it.
